# Test the UCloud model
Run top to bottom in an existing Python 3.11+ Jupyter kernel. The first cell installs client libraries, not Jupyter. This uses your UCloud endpoint and `VLLM_API_KEY` from the repository `.env`.
The hosting job must be running; this notebook does not start or extend it.
Client reference: [OpenAI Python SDK](https://github.com/openai/openai-python). Server reference: [vLLM 0.31.0 API](https://docs.vllm.ai/en/v0.31.0/serving/online_serving/openai_compatible_server/).

In [ ]:
# Install into the notebook kernel. Versions match the tested client.
%pip install -q openai==3.24.0 python-dotenv==1.2.4

## Connect
The project is found from the current directory or its parents. If you moved the notebook elsewhere, set `PROJECT_ROOT` explicitly. Set `PROFILE_FILE` to your reviewed local profile, for example `config/local.qwen.json`; its request.name must match the deployed alias.
The notebook prefers local root `.env` values over inherited environment variables, unlike the deployment CLI. Process-only CLI overrides do not configure it. Keep credentials out of cells and clear outputs before sharing or committing.

In [ ]:
import json
import os
from pathlib import Path
from time import perf_counter
from urllib.parse import urlsplit
from dotenv import dotenv_values
from openai import OpenAI, DefaultHttpx2Client, APIConnectionError, APIStatusError

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'config' / 'hosting.qwen-small.json').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Set PROJECT_ROOT to the UCloud vLLM agent repository folder.')
PROFILE_FILE = PROJECT_ROOT / 'config' / 'hosting.qwen-small.json'
MODEL = json.loads(PROFILE_FILE.read_text(encoding='utf-8'))['request']['name']
settings = dotenv_values(PROJECT_ROOT / '.env', interpolate=False)
api_key = settings.get('VLLM_API_KEY') or os.getenv('VLLM_API_KEY')
base_url = settings.get('UCLOUD_PUBLIC_BASE_URL') or os.getenv('UCLOUD_PUBLIC_BASE_URL')
if not api_key or not base_url:
    raise RuntimeError('Set VLLM_API_KEY and UCLOUD_PUBLIC_BASE_URL in the repository .env.')
base_url = base_url.strip().rstrip('/')
if '://' not in base_url:
    base_url = 'https://' + base_url
parts = urlsplit(base_url)
if (parts.scheme != 'https' or not parts.hostname or parts.username or parts.password
        or parts.query or parts.fragment):
    raise RuntimeError('Use a plain HTTPS API URL without credentials, query or fragment.')
if not parts.path.rstrip('/').endswith('/v1'):
    base_url += '/v1'
client = OpenAI(base_url=base_url, api_key=api_key, timeout=30.0, max_retries=0,
                http_client=DefaultHttpx2Client(follow_redirects=False))

def call_api(operation, **kwargs):
    try:
        return operation(**kwargs)
    except APIConnectionError:
        raise RuntimeError('Cannot reach the model. Check the URL and UCloud job state.') from None
    except APIStatusError as error:
        raise RuntimeError(f'API returned HTTP {error.status_code}. Check the key and model readiness.') from None

print(f'Endpoint: {base_url}')
print(f'Model: {MODEL}')

## 1. Check model availability

In [ ]:
available_models = [model.id for model in call_api(client.models.list).data]
print('Available models:', available_models)
assert MODEL in available_models, f'Expected model {MODEL!r} is not available.'
print('PASS: expected model is available')

## 2. Send a chat prompt
Edit `PROMPT` to ask your own question. Temperature is zero; assertions check connectivity and response structure rather than exact wording.

In [ ]:
PROMPT = 'What is 2 + 2? Answer briefly.'
started = perf_counter()
response = call_api(client.chat.completions.create, model=MODEL,
                    messages=[{'role': 'user', 'content': PROMPT}],
                    temperature=0, max_tokens=128)
assert response.model == MODEL, 'Response model differs from the requested alias.'
assert response.choices, 'Response has no choices.'
answer = response.choices[0].message.content
assert answer and answer.strip(), 'Model returned an empty answer.'
assert response.choices[0].finish_reason in ('stop', 'length'), 'Response did not finish normally.'
print(answer)
print(f'PASS: chat response completed in {perf_counter() - started:.2f}s')

## 3. Stream a response

In [ ]:
pieces = []
finish_reason = None
started = perf_counter()
with call_api(client.chat.completions.create, model=MODEL,
              messages=[{'role': 'user', 'content': 'Explain in two sentences what a GPU does.'}],
              temperature=0, max_tokens=128, stream=True) as stream:
    for chunk in stream:
        assert chunk.model == MODEL, 'Stream model differs from the requested alias.'
        for choice in chunk.choices:
            if choice.delta.content:
                pieces.append(choice.delta.content)
                print(choice.delta.content, end='', flush=True)
            if choice.finish_reason is not None:
                finish_reason = choice.finish_reason
assert ''.join(pieces).strip(), 'Stream returned no text.'
assert finish_reason in ('stop', 'length'), 'Stream did not finish normally.'
print()
print(f'PASS: streaming completed in {perf_counter() - started:.2f}s')

## 4. Check authentication
A deliberately incorrect key must be rejected with HTTP 401 or 403.

In [ ]:
from secrets import token_urlsafe

with OpenAI(base_url=base_url, api_key='invalid-test-' + token_urlsafe(24),
            timeout=15.0, max_retries=0,
            http_client=DefaultHttpx2Client(follow_redirects=False)) as invalid_client:
    try:
        invalid_client.models.list()
    except APIStatusError as error:
        assert error.status_code in (401, 403), f'Unexpected HTTP status: {error.status_code}'
        print(f'PASS: incorrect key rejected (HTTP {error.status_code})')
    else:
        raise AssertionError('Endpoint accepted an incorrect API key.')
client.close()

If a test fails: **401/403** indicates authentication trouble; **404** can indicate an incorrect API path or model alias; **502/503** can mean the model is starting, its process failed, or the allocation expired. Re-run Connect after changing `.env` or to reopen the client after the last test.
For the agent's complete seven-check report, use the repository CLI `check` command. Keep credentials out of notebook cells and outputs.